### 2 - Teste Cálculos Indicadores

#### 2.1 Importações e carga dos dados tratados:

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Dimensão original df_dados_tratados:", df_dados_tratados.shape)

Dimensão original df_dados_tratados: (10445, 19)


In [2]:
df_dados_tratados.head()

,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Concluinte_Previsto
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino,False
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
3,2017,Em curso,Branca,66262151,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
4,2017,Evadidos,Branca,66262139,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2017-05-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Transf. externa,Elétrica,Técnico-Integrado,Matutino,False


In [3]:
df_dados_tratados.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

#### 2.2 Cálculos dos Indicadores de Permanência e Êxito:

In [ ]:
def calcular_indicadores(df, agrupamento):

    df_indicadores = (
        df.groupby(agrupamento)
        .agg(
            ingressantes    = ("Situação de Matrícula", lambda x: (x == "Ingressante").sum()),
            em_curso        = ("Categoria da Situação",  lambda x: (x == "Em curso").sum()),
            concluintes     = ("Categoria da Situação",  lambda x: (x == "Concluintes").sum()),
            evadidos        = ("Categoria da Situação",  lambda x: (x == "Evadidos").sum()),
            matr_atendidas  = ("Código da Matricula",    "count"),
            conc_no_prazo   = ("Situação de Matrícula",  lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula",  lambda x: (x == "Concluída com atraso").sum()),
            abandono        = ("Situação de Matrícula",  lambda x: (x == "Abandono").sum()),
            desligados      = ("Situação de Matrícula",  lambda x: (x == "Desligada").sum()),
            transf_ext      = ("Situação de Matrícula",  lambda x: (x == "Transf. externa").sum()),
            transf_int      = ("Situação de Matrícula",  lambda x: (x == "Transf. interna").sum()),
            integralizadas  = ("Situação de Matrícula",  lambda x: (x == "Integralizada").sum()),
            conc_previstos  = ("Concluinte_Previsto",    "sum"),
            MREG            = ("Situação de Matrícula",
                               lambda x: ((x == "Em fluxo") | (x == "Ingressante")).sum()),
            MRET            = ("Situação de Matrícula",  lambda x: (x == "Retido").sum()),
        )
        .reset_index()
    )

    # Evita divisão por zero substituindo 0 por NaN antes de dividir
    ma = df_indicadores["matr_atendidas"].replace(0, np.nan)

    # TC — Taxa de Conclusão
    df_indicadores["TC"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / ma * 100
    )

    # TE — Taxa de Evasão
    df_indicadores["TE"] = (
        (df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"])
        / ma * 100
    )

    # TR — Taxa de Retenção
    df_indicadores["TR"] = df_indicadores["MRET"] / ma * 100

    # IEf — Índice de Eficiência
    # Numerador: todos os concluintes (no prazo + com atraso)
    # Denominador: matrículas que tiveram algum desfecho (todas as saídas)
    matr_finalizadas = (
        df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]
        + df_indicadores["abandono"] + df_indicadores["desligados"]
        + df_indicadores["transf_int"] + df_indicadores["transf_ext"]
    ).replace(0, np.nan)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"])
        / matr_finalizadas * 100
    )

    # t_matr_cont_reg — Taxa de Matrícula Continuada Regular - para o cálculo da TPE
    df_indicadores["t_matr_cont_reg"] = df_indicadores["MREG"] / ma * 100

    # TPE — Taxa de Permanência e Êxito
    df_indicadores["TPE"] = df_indicadores["TC"] + df_indicadores["t_matr_cont_reg"]

    # TEFAcad — Taxa de Efetividade Acadêmica
    df_indicadores["TEFAcad"] = (
        df_indicadores["conc_no_prazo"]
        / df_indicadores["conc_previstos"].replace(0, np.nan) * 100
    )

    return df_indicadores.fillna(0).round(2)


# Calcula por Ano e por Ano + Curso
ind_ano       = calcular_indicadores(df_dados_tratados, ["Ano"])
ind_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])

print(ind_ano[["Ano", "TC", "TE", "TR", "TPE", "TEFAcad", "IEf"]])


    Ano    TC     TE     TR    TPE  TEFAcad    IEf
0  2017  8.38  15.78   5.18  78.55    45.71  34.34
1  2018  7.80   9.84   7.70  82.36    43.23  43.96
2  2019  6.07  11.05   5.68  83.19    26.87  35.45
3  2020  0.16   7.52  13.57  78.66     0.00   2.06
4  2021  2.50   5.70  23.89  70.41     0.00  30.43
5  2022  7.48   9.29  34.60  56.11    15.77  44.62
6  2023  9.77   5.27  31.94  62.79    47.78  64.94
7  2024  5.91   2.32  35.15  62.42    26.42  71.81


In [8]:
df_indicadores_ano = calcular_indicadores(df_dados_tratados, ["Ano"])
df_indicadores_ano.head(20)

,Ano,ingressantes,em_curso,concluintes,evadidos,matr_atendidas,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int,integralizadas,conc_previstos,MREG,MRET,TC,TE,TR,IEf,t_matr_cont_reg,TPE,TEFAcad
0,2017,292,611,70,130,811,48,20,70,45,13,2,2,105,569,42,8.38,15.78,5.18,34.34,70.16,78.55,45.71
1,2018,344,844,80,102,1026,67,13,40,46,15,1,0,155,765,79,7.80,9.84,7.70,43.96,74.56,82.36,43.23
2,2019,373,1064,79,142,1285,61,17,103,33,6,0,1,227,991,73,6.07,11.05,5.68,35.45,77.12,83.19,26.87
3,2020,218,1126,2,95,1223,0,2,78,10,4,3,0,285,960,166,0.16,7.52,13.57,2.06,78.50,78.66,0.00
4,2021,147,1030,28,64,1122,0,28,27,18,19,0,0,319,762,268,2.50,5.70,23.89,30.43,67.91,70.41,0.00
5,2022,323,1246,112,139,1497,50,62,83,36,20,0,0,317,728,518,7.48,9.29,34.60,44.62,48.63,56.11,15.77
6,2023,425,1418,163,88,1669,97,66,47,23,18,0,0,203,885,533,9.77,5.27,31.94,64.94,53.03,62.79,47.78
7,2024,358,1661,109,42,1812,56,51,1,33,8,0,2,212,1024,637,5.91,2.32,35.15,71.81,56.51,62.42,26.42


In [9]:
df_indicadores_ano.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Ano              8 non-null      int64  
 1   ingressantes     8 non-null      int64  
 2   em_curso         8 non-null      int64  
 3   concluintes      8 non-null      int64  
 4   evadidos         8 non-null      int64  
 5   matr_atendidas   8 non-null      int64  
 6   conc_no_prazo    8 non-null      int64  
 7   conc_com_atraso  8 non-null      int64  
 8   abandono         8 non-null      int64  
 9   desligados       8 non-null      int64  
 10  transf_ext       8 non-null      int64  
 11  transf_int       8 non-null      int64  
 12  integralizadas   8 non-null      int64  
 13  conc_previstos   8 non-null      int64  
 14  MREG             8 non-null      int64  
 15  MRET             8 non-null      int64  
 16  TC               8 non-null      float64
 17  TE               8 n

In [11]:
df_indicadores_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])
df_indicadores_ano_curso.head(10)

,Ano,Nome de Curso,ingressantes,em_curso,concluintes,evadidos,matr_atendidas,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int,integralizadas,conc_previstos,MREG,MRET,TC,TE,TR,IEf,t_matr_cont_reg,TPE,TEFAcad
0,2017,Análise e Desenvolvimento de Sistemas,59,194,17,36,247,5,12,25,8,2,1,0,27,163,31,6.88,14.17,12.55,32.08,65.99,72.87,18.52
1,2017,Eletrônica Industrial,27,59,0,25,84,0,0,19,6,0,0,0,11,59,0,0.00,29.76,0.00,0.00,70.24,70.24,0.00
2,2017,Gestão Desportiva e de Lazer,29,80,14,18,112,8,4,13,5,0,0,2,20,74,6,10.71,16.07,5.36,40.00,66.07,76.79,40.00
3,2017,Letras - Português e Espanhol,25,25,0,5,30,0,0,0,5,0,0,0,0,25,0,0.00,16.67,0.00,0.00,83.33,83.33,0.00
4,2017,Técnico em Comércio,33,33,0,0,33,0,0,0,0,0,0,0,0,33,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
5,2017,Técnico em Eletrônica,18,80,14,21,115,11,3,4,10,6,1,0,23,76,4,12.17,17.39,3.48,40.00,66.09,78.26,47.83
6,2017,Técnico em Guia de Turismo,34,43,25,19,87,24,1,9,10,0,0,0,24,42,1,28.74,21.84,1.15,56.82,48.28,77.01,100.00
7,2017,Técnico em Informática,46,46,0,2,48,0,0,0,0,2,0,0,0,46,0,0.00,4.17,0.00,0.00,95.83,95.83,0.00
8,2017,Técnico em Lazer,21,51,0,4,55,0,0,0,1,3,0,0,0,51,0,0.00,7.27,0.00,0.00,92.73,92.73,0.00
9,2018,Análise e Desenvolvimento de Sistemas,67,227,14,26,267,10,4,15,11,0,0,0,46,173,54,5.24,9.74,20.22,35.00,64.79,70.04,21.74


In [12]:
df_indicadores_ano_tipo_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Tipo de Curso"])
df_indicadores_ano_tipo_curso.head()

,Ano,Tipo de Curso,ingressantes,em_curso,concluintes,evadidos,matr_atendidas,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int,integralizadas,conc_previstos,MREG,MRET,TC,TE,TR,IEf,t_matr_cont_reg,TPE,TEFAcad
0,2017,Superior-Licenciatura,25,25,0,5,30,0,0,0,5,0,0,0,0,25,0,0.00,16.67,0.00,0.00,83.33,83.33,0.00
1,2017,Superior-Tecnologia,115,333,31,79,443,13,16,57,19,2,1,2,58,296,37,6.55,17.61,8.35,26.85,66.82,73.36,22.41
2,2017,Técnico-Integrado,85,177,14,27,218,11,3,4,11,11,1,0,23,173,4,6.42,11.93,1.83,34.15,79.36,85.78,47.83
3,2017,Técnico-PROEJA - Integrado,33,33,0,0,33,0,0,0,0,0,0,0,0,33,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
4,2017,Técnico-Subsequente,34,43,25,19,87,24,1,9,10,0,0,0,24,42,1,28.74,21.84,1.15,56.82,48.28,77.01,100.00


In [13]:
df_indicadores_ano_eixo_tecnologico = calcular_indicadores(df_dados_tratados, ["Ano", "Eixo Tecnológico"])
df_indicadores_ano_eixo_tecnologico.head()

,Ano,Eixo Tecnológico,ingressantes,em_curso,concluintes,evadidos,matr_atendidas,conc_no_prazo,conc_com_atraso,abandono,desligados,transf_ext,transf_int,integralizadas,conc_previstos,MREG,MRET,TC,TE,TR,IEf,t_matr_cont_reg,TPE,TEFAcad
0,2017,Controle e Processos Industriais,45,139,14,46,199,11,3,23,16,6,1,0,34,135,4,7.04,22.61,2.01,23.33,67.84,74.87,32.35
1,2017,Desenvolvimento Educacional e Social,25,25,0,5,30,0,0,0,5,0,0,0,0,25,0,0.00,16.67,0.00,0.00,83.33,83.33,0.00
2,2017,Gestão e Negócios,33,33,0,0,33,0,0,0,0,0,0,0,0,33,0,0.00,0.00,0.00,0.00,100.00,100.00,0.00
3,2017,Informação e Comunicação,105,240,17,38,295,5,12,25,8,4,1,0,27,209,31,5.76,12.54,10.51,30.91,70.85,76.61,18.52
4,2017,"Turismo, Hospitalidade e Lazer",84,174,39,41,254,32,5,22,16,3,0,2,44,167,7,14.57,16.14,2.76,47.44,65.75,80.31,72.73


In [ ]:
#df_indicadores_ano.to_parquet("teste_indicadores_ano.parquet", index=False)
#df_indicadores_ano_curso.to_parquet("teste_indicadores_curso_ano.parquet", index=False)
#df_indicadores_ano_tipo_curso.to_parquet("teste_indicadores_ano_tipo_curso.parquet", index=False)
#df_indicadores_ano_eixo_tecnologico.to_parquet("teste_indicadores_ano_tipo_curso.parquet", index=False)